In [ ]:
# KALMAN GOLD-20 RECOVERY FORENSICS v2.8 — ONE CELL / READ ONLY
# v2.4 legitimately updated revalidation_data_ready to 888, so it can no longer identify the original gold-20.
# Recover the original gold rows from the PRE-v2.4 backup where derived fields existed before expansion.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
from pathlib import Path
import pandas as pd, numpy as np, glob, os

ROOT=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results/open_revalidation_v1")
AUD=ROOT/"open_revalidation_trade_audit.parquet"
backs=sorted(ROOT.glob("open_revalidation_trade_audit.pre_derived_v2_4_*.parquet"))
if not backs: raise RuntimeError("No pre-v2.4 backup found")
PRE=backs[-1]
a=pd.read_parquet(AUD); pre=pd.read_parquet(PRE)
print("[CURRENT]",AUD,a.shape)
print("[PRE-V2.4]",PRE,pre.shape)

# Original gold-20 = rows where the pre-v2.4 reconstructed IEX net already existed.
goldmask=pre["reconstructed_fixed4_net_return"].notna()
print("[RECOVERED GOLD]",int(goldmask.sum()))
if int(goldmask.sum())!=20: raise RuntimeError(f"Expected 20 legacy-derived rows in pre-v2.4 backup, got {int(goldmask.sum())}")

keys=["fold","symbol","entry_timestamp","entry_seq","exit_seq"]
keys=[k for k in keys if k in a.columns and k in pre.columns]
gold_keys=pre.loc[goldmask,keys].drop_duplicates()
gold=a.merge(gold_keys,on=keys,how="inner")
print("[CURRENT GOLD JOIN]",gold.shape)
if len(gold)!=20: raise RuntimeError(f"Gold key join expected 20, got {len(gold)}")

policies=sorted({c[:-9] for c in a.columns if c.startswith("OPEN_") and c.endswith("__trigger")})
features=["position_return_prev_close","position_return_open","position_return_5m","position_return_15m",
          "overnight_gap_return","open_momentum_5m","open_momentum_15m","giveback_prev_close_to_5m"]
features=[c for c in features if c in gold.columns]
print("[POLICIES]",policies)

# Use trigger/net/exit labels from PRE-v2.4 backup, not possibly refreshed current values.
labelcols=[]
for p in policies:
    labelcols += [p+"__trigger",p+"__net_return",p+"__exit_price"]
lab=pre.loc[goldmask,keys+labelcols].copy()
gold=gold.drop(columns=[c for c in labelcols if c in gold.columns]).merge(lab,on=keys,how="left")
print("[TRIGGER COUNTS]")
for p in policies: print(p,int(gold[p+"__trigger"].fillna(False).astype(bool).sum()))

atoms=[]
for c in features:
    x=pd.to_numeric(gold[c],errors="coerce")
    vals=np.sort(x.dropna().unique())
    mids=[(vals[i]+vals[i+1])/2 for i in range(len(vals)-1)]
    ts=sorted(set([0.0,-0.002,0.002]+list(vals)+mids))
    for t in ts:
        for op in ("<=","<",">=",">"):
            if op=="<=": pred=(x<=t)
            elif op=="<": pred=(x<t)
            elif op==">=": pred=(x>=t)
            else: pred=(x>t)
            atoms.append((f"{c}{op}{t:.12g}",pred.fillna(False)))

def metric(y,p):
    y=np.asarray(y,bool); p=np.asarray(p,bool)
    fp=int((~y&p).sum()); fn=int((y&~p).sum())
    return fp+fn,fp,fn,int(p.sum())

print("\n[EXACT GOLD-20 SEARCH]")
for pol in policies:
    y=gold[pol+"__trigger"].fillna(False).astype(bool)
    print("\n###",pol,"actual_true=",int(y.sum()))
    if y.sum()==0:
        print("UNIDENTIFIABLE: zero positive gold labels; source/spec required.")
        continue
    singles=[]
    for n,p in atoms:
        e,fp,fn,nt=metric(y,p); singles.append((e,1,n,fp,fn,nt,p))
    seed=sorted(singles,key=lambda z:(z[0],abs(z[5]-int(y.sum())),len(z[2])))[:120]
    ranked=list(singles)
    for i,z1 in enumerate(seed):
        for z2 in seed[i+1:]:
            for op in ("AND","OR"):
                p=(z1[6]&z2[6]) if op=="AND" else (z1[6]|z2[6])
                e,fp,fn,nt=metric(y,p)
                ranked.append((e,2,f"({z1[2]}) {op} ({z2[2]})",fp,fn,nt,p))
    ranked=sorted(ranked,key=lambda z:(z[0],z[1],len(z[2])))
    for z in ranked[:12]:
        print("err=",z[0],"complexity=",z[1],"fp=",z[3],"fn=",z[4],"true=",z[5],z[2])
    exact=[]; seen=set()
    for z in ranked:
        if z[0]!=0: break
        if z[2] not in seen: exact.append(z[2]); seen.add(z[2])
        if len(exact)>=12: break
    print("[EXACT]",len(exact))
    for s in exact: print(" ",s)

print("\n[BOUNDARY TABLE]")
for pol in policies:
    y=gold[pol+"__trigger"].fillna(False).astype(bool)
    if not y.any(): continue
    print("\n",pol)
    for c in features:
        x=pd.to_numeric(gold[c],errors="coerce"); pos=x[y].dropna(); neg=x[~y].dropna()
        print(c,"POS", [float(pos.min()),float(pos.max())] if len(pos) else None,
              "NEG",[float(neg.min()),float(neg.max())] if len(neg) else None)

print("\n[CONTRACT CHECK]")
exit_map={"OPEN_NEG_0BP_0M":"open_0_price_iex","OPEN_NEG_20BP_0M":"open_0_price_iex","OPEN_FLIP_0M":"open_0_price_iex",
"OPEN_NEG_5M_CONFIRM":"open_5_price_iex","OPEN_FLIP_5M_CONFIRM":"open_5_price_iex","OPEN_GAP_5M_CONFIRM":"open_5_price_iex",
"OPEN_GIVEBACK_5M":"open_5_price_iex","OPEN_NEG_15M_CONFIRM":"open_15_price_iex"}
ret_map={"open_0_price_iex":"position_return_open","open_5_price_iex":"position_return_5m","open_15_price_iex":"position_return_15m"}
for p in policies:
    y=gold[p+"__trigger"].fillna(False).astype(bool)
    print("\n",p,"trigger=",int(y.sum()))
    if y.any():
        ep=pd.to_numeric(gold.loc[y,p+"__exit_price"],errors="coerce"); px=pd.to_numeric(gold.loc[y,exit_map[p]],errors="coerce")
        nr=pd.to_numeric(gold.loc[y,p+"__net_return"],errors="coerce")
        rr=pd.to_numeric(gold.loc[y,ret_map[exit_map[p]]],errors="coerce")*pd.to_numeric(gold.loc[y,"weight"],errors="coerce")-pd.to_numeric(gold.loc[y,"cost_proxy"],errors="coerce")
        print("exit_maxerr",float((ep-px).abs().max()),"net_maxerr",float((nr-rr).abs().max()))
    nt=~y
    nr=pd.to_numeric(gold.loc[nt,p+"__net_return"],errors="coerce"); hist=pd.to_numeric(gold.loc[nt,"net_return"],errors="coerce")
    print("nontrigger_vs_historical_fixed4_maxerr",float((nr-hist).abs().max()) if len(nr) else None)

print("\nREAD ONLY. No canonical file modified.")
print("NEXT: use exact gold-20 evidence plus original source/spec; do not promote ambiguous predicates.")
